# 08 — Concordancia estructural externa bloqueada

Evalúa únicamente las **284 aristas dirigidas del núcleo Basal-like 5/5** ya fijado. No carga pesos, no reentrena, no vuelve a seleccionar el grafo y no modifica ningún hiperparámetro.

El contraste principal es la correlación de Spearman entre el efecto de correlación diferencial Basal-like frente al resto obtenido en desarrollo y el observado externamente. Las concordancias de signo son secundarias. Cada estadístico se compara con 1.000 paneles del prior emparejados por tipo, signo y bins de grado dirigido; las aristas observadas se excluyen del nulo.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'intergate').is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Ejecute este notebook dentro del repositorio InterGATE.')
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from intergate.config import CFG
from intergate.runtime_setup import (
    assert_transport_protocol, assert_v6_protocol, prepare_runtime,
)
from intergate.posthoc_external_concordance import (
    run_external_structural_concordance,
)

assert_v6_protocol(CFG)
assert_transport_protocol(CFG)

In [ ]:
ARTIFACTS = Path(CFG.ARTIFACTS_ROOT)
RUN_ROOT = ARTIFACTS / 'INTERGATE_V6'
RESULTS = PROJECT_ROOT / 'results_v6'
LOCKED_SEED_TABLES = (
    RESULTS / '04_controls' / 'locked_biology_score_w0075' / 'seed_tables'
)
PROTOCOL_LOCK = ARTIFACTS / 'protocol_lock_v6.json'
EXTERNAL_COMPLETION = (
    RESULTS / '07_external_locked' / 'external_evaluation_COMPLETE.json'
)
OUT = RESULTS / '08_external_structural_concordance'

N_MATCHED_DRAWS = 1000
RANDOM_SEED = 20260821

if not EXTERNAL_COMPLETION.is_file():
    raise FileNotFoundError('Ejecute primero 07_test_externo_bloqueado.ipynb.')
print('Cohortes externas:', tuple(CFG.TEST_COHORTS))
print('Nulos emparejados:', N_MATCHED_DRAWS)
print('Salida:', OUT)

In [ ]:
# Única apertura del runtime externo en este análisis post-lock.
runtime = prepare_runtime(CFG, include_external=True)
assert runtime['external_opened'] is True

external = run_external_structural_concordance(
    runtime,
    run_root=RUN_ROOT,
    locked_seed_tables=LOCKED_SEED_TABLES,
    output_dir=OUT,
    external_cohorts=tuple(CFG.TEST_COHORTS),
    protocol_lock=PROTOCOL_LOCK,
    external_completion=EXTERNAL_COMPLETION,
    n_draws=N_MATCHED_DRAWS,
    random_seed=RANDOM_SEED,
    batch_size=int(CFG.BIOLOGY_EDGE_BATCH_SIZE),
)
print('Análisis externo estructural completo.')

In [ ]:
display(external['cohort_counts'])
primary = external['summary'][
    external['summary'].is_primary_metric
].copy()
display(primary[[
    'scope', 'observed', 'null_mean', 'null_q025', 'null_q975',
    'empirical_p_upper', 'fdr_primary_four_scopes',
]])
display(external['summary'][[
    'scope', 'metric', 'observed', 'null_mean',
    'empirical_p_upper', 'fdr_all_12_tests',
]])

In [ ]:
# Figura descriptiva del contraste principal; los p/FDR están en la tabla.
plot = primary.reset_index(drop=True)
ypos = np.arange(len(plot))
fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.hlines(ypos, plot.null_q025, plot.null_q975, color='#8c8c8c', lw=5,
          label='95% de paneles emparejados')
ax.scatter(plot.null_mean, ypos, color='#4d4d4d', s=45, zorder=3,
           label='media nula')
ax.scatter(plot.observed, ypos, color='#b2182b', s=75, zorder=4,
           label='núcleo Basal-like 5/5')
ax.axvline(0, color='black', lw=0.8, alpha=0.5)
ax.set_yticks(ypos, plot.scope)
ax.set_xlabel('Spearman: efecto diferencial de desarrollo vs externo')
ax.set_title('Concordancia estructural externa del núcleo Basal-like')
ax.legend(frameon=False, loc='best')
fig.tight_layout()
OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / 'external_structural_concordance.pdf', bbox_inches='tight')
fig.savefig(OUT / 'external_structural_concordance.png', dpi=600,
            bbox_inches='tight')
plt.show()

## Regla de interpretación

La transferencia biológica se considerará respaldada si el estadístico principal del metaanálisis externo es positivo y supera el nulo emparejado tras corregir los cuatro ámbitos externos. Las cohortes individuales se mostrarán siempre, aunque difieran entre sí. Una concordancia de signo secundaria no sustituirá al contraste principal. El análisis no autoriza causalidad ni descubrimiento de interacciones fuera del prior.